In [ ]:
# Pandas, Matplotlib, OS, SciPy for data analysis and visualization
import pandas as pd
import matplotlib.pyplot as plt
import os
from scipy import stats


In [ ]:
# variants configuration
variants = {
    "Baseline": {
        "path": r"C:\Users\User\Desktop\semestre_2\Inteligencia_Computacional_Otimizacao\CIFO_Group37_Vestigial\Project\data\results\Baseline\statistical_summary.csv",
        "color": "blue",
        "x_col": "Generation",
    },
    "Crossover Variant (Two-Point)": {
        "path": r"C:\Users\User\Desktop\semestre_2\Inteligencia_Computacional_Otimizacao\CIFO_Group37_Vestigial\Project\data\results\Crossover_Variant_TwoPoint\statistical_summary.csv",
        "color": "green",
        "x_col": "Generation",
    },
    "Mutation Variant (Inversion)": {
        "path": r"C:\Users\User\Desktop\semestre_2\Inteligencia_Computacional_Otimizacao\CIFO_Group37_Vestigial\Project\data\results\Mutation_Variant_Inversion\statistical_summary.csv",
        "color": "red",
        "x_col": "Generation",
    },
}

output_dir = r"C:\Users\User\Desktop\semestre_2\Inteligencia_Computacional_Otimizacao\CIFO_Group37_Vestigial\Project\data\results\Plots"
os.makedirs(output_dir, exist_ok=True)

In [ ]:
def read_csv_clean(path):
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip()  # Remove whitespace from column names
    return df

In [ ]:
def normalize_x(df, x_col):
    # Normalizes the x-axis to a percentage scale (0-100%) based on the range of the x_col values
    x = df[x_col]
    return (x - x.min()) / (x.max() - x.min()) * 100

In [ ]:
def plot_variants(metric_col, ylabel, title, filename, show_std=False, normalize=True):
    fig, ax = plt.subplots(figsize=(12, 6))

    for label, cfg in variants.items():
        # Pre-filter: only process if the file exists
        if not os.path.exists(cfg["path"]):
            continue

        df = read_csv_clean(cfg["path"])
        
        # Assume x_col is always valid if you pre-validate the config dict
        x = normalize_x(df, cfg["x_col"]) if normalize else df[cfg["x_col"]]

        ax.plot(x, df[metric_col], label=label, color=cfg["color"], linewidth=2)

        # Check if Std_Dev column exists before trying to plot it
        if show_std and "Std_Dev" in df.columns:
            ax.fill_between(x, 
                            df[metric_col] - df["Std_Dev"], 
                            df[metric_col] + df["Std_Dev"], 
                            color=cfg["color"], alpha=0.15)

    # Check if anything was actually plotted
    if not ax.has_data():
        print(f"No valid data found for '{title}'. Skipping plot.")
        plt.close()
        return

    # Set labels
    xlabel = "Evolution Progress (%)\n(GA: Generations | SA: Iterations)" if normalize else "Steps"
    ax.set(title=title, xlabel=xlabel, ylabel=ylabel)
    ax.legend(loc="upper right")
    ax.grid(True, linestyle="--", alpha=0.6)
    fig.tight_layout()

    # Save and cleanup
    fig.savefig(os.path.join(output_dir, filename), dpi=300)
    plt.close()
    print(f"Plot saved: {filename}")

In [6]:
plot_variants(
    metric_col="Mean_RMSE",
    ylabel="Mean RMSE",
    title="Variant Comparison (Mean RMSE)",
    filename="mean_variant_comparison.png",
    show_std=True,
    normalize=True
)

plot_variants(
    metric_col="Best_RMSE",
    ylabel="RMSE",
    title="Variant Comparison (Best Solution)",
    filename="best_variant_comparison.png",
    show_std=False,
    normalize=True
)

Plot saved in : C:\Users\User\Desktop\semestre_2\Inteligencia_Computacional_Otimizacao\CIFO_Group37_Vestigial\Project\data\results\Plots\mean_variant_comparison.png
Plot saved in : C:\Users\User\Desktop\semestre_2\Inteligencia_Computacional_Otimizacao\CIFO_Group37_Vestigial\Project\data\results\Plots\best_variant_comparison.png


In [7]:
variants = {
            "Simulated Annealing": {
        "path": r"C:\Users\User\Desktop\semestre_2\Inteligencia_Computacional_Otimizacao\CIFO_Group37_Vestigial\Project\data\results\Simulated_Annealing\statistical_summary.csv",
        "color": "red",
        "x_col": "Iteration"
    },
            "Best Variant (Mutation: Inversion)": {
        "path": r"C:\Users\User\Desktop\semestre_2\Inteligencia_Computacional_Otimizacao\CIFO_Group37_Vestigial\Project\data\results\Mutation_Variant_Inversion\statistical_summary.csv",
        "color": "blue",
        "x_col": "Generation",
    },
}

In [8]:
plot_variants(
    metric_col="Mean_RMSE",
    ylabel="Mean RMSE",
    title="Challenge 2: GA Vs SA (Mean RMSE)",
    filename="mean_c2_comparison.png",
    show_std=True
)

plot_variants(
    metric_col="Best_RMSE",
    ylabel="RMSE",
    title="Challenge 2: GA Vs SA (Best Solution)",
    filename="best_c2_comparison.png",
    show_std=False
)

Plot saved in : C:\Users\User\Desktop\semestre_2\Inteligencia_Computacional_Otimizacao\CIFO_Group37_Vestigial\Project\data\results\Plots\mean_c2_comparison.png
Plot saved in : C:\Users\User\Desktop\semestre_2\Inteligencia_Computacional_Otimizacao\CIFO_Group37_Vestigial\Project\data\results\Plots\best_c2_comparison.png


## Crossover Vs Mutation Variant

In [ ]:
# Path for the comparison between variants
path_mutation_variant  = r"C:\Users\User\Desktop\semestre_2\Inteligencia_Computacional_Otimizacao\CIFO_Group37_Vestigial\Project\data\results\Mutation_Variant_Inversion\raw_fitness_data_50.csv"
path_crossover_variant = r"C:\Users\User\Desktop\semestre_2\Inteligencia_Computacional_Otimizacao\CIFO_Group37_Vestigial\Project\data\results\Crossover_Variant_TwoPoint\raw_fitness_data_50.csv"

# Load the results for variants
mutation_variant_data  = pd.read_csv(path_mutation_variant,  index_col=0)
crossover_variant_data = pd.read_csv(path_crossover_variant, index_col=0)

# Extract the final RMSE values (in the last column) for both variants
mutation_variant_final  = mutation_variant_data.iloc[:, -1].astype(float)
crossover_variant_final = crossover_variant_data.iloc[:, -1].astype(float)

print(f"Mutation Variant final RMSE: mean={mutation_variant_final.mean():.4f}  std={mutation_variant_final.std():.4f}")
print(f"Crossover Variant final RMSE: mean={crossover_variant_final.mean():.4f}  std={crossover_variant_final.std():.4f}")

# Perform Mann-Whitney U test to determine if the difference is statistically significant
# We use alternative = 'less' because its a minimization problem and we want to test if the mutation variant is better than the crossover variant (lower RMSE)
u_stat, p_val = stats.mannwhitneyu(mutation_variant_final, crossover_variant_final, alternative='less')

print(f"\nStatistical Test: Mann-Whitney U")
print(f"U-statistic: {u_stat:.4f}")
print(f"p-value:     {p_val:.5f}")
# Check if the p-value is less than the significance level (0.05)
if p_val < 0.05:
    print("There is STATISTICALLY SIGNIFICANT evidence that Mutation variant performs better than Crossover variant (Mutation < Crossover).")
else:
    print("No statistical evidence of difference between Mutation and Crossover variants.")

# Calculate the percentage of runs that met the convergence threshold (RMSE < 30.0)
limiar = 30.0
sr_mutation  = (mutation_variant_final  < limiar).mean()
sr_crossover = (crossover_variant_final < limiar).mean()

print(f"\nSuccess Rate (RMSE < {limiar})")
print(f"Mutation Variant:  {sr_mutation:.2%}")
print(f"Crossover Variant: {sr_crossover:.2%}")

Mutation Variant final RMSE: mean=30.4000  std=1.6746
Crossover Variant final RMSE: mean=35.7950  std=1.8973

Statistical Test: Mann-Whitney U
U-statistic: 9.0000
p-value:     0.00000
There is STATISTICALLY SIGNIFICANT evidence that Mutation variant performs better than Crossover variant (Mutation < Crossover).

Success Rate (RMSE < 30.0)
Mutation Variant:  55.00%
Crossover Variant: 0.00%


## Baseline Vs Best Variant (Mutation_Variant_TwoPoint)

In [ ]:
# Path for the comparison between baseline and the best variant
path_baseline     = r"C:\Users\User\Desktop\semestre_2\Inteligencia_Computacional_Otimizacao\CIFO_Group37_Vestigial\Project\data\results\Baseline\raw_fitness_data_50.csv"
path_best_variant = r"C:\Users\User\Desktop\semestre_2\Inteligencia_Computacional_Otimizacao\CIFO_Group37_Vestigial\Project\data\results\Mutation_Variant_Inversion\raw_fitness_data_50.csv"

# Load the results for baseline and best variant
baseline_data     = pd.read_csv(path_baseline,     index_col=0)
best_variant_data = pd.read_csv(path_best_variant, index_col=0)

# Extract the final RMSE values (in the last column) for variant and baseline
baseline_final     = baseline_data.iloc[:, -1].astype(float)
best_variant_final = best_variant_data.iloc[:, -1].astype(float) 

print(f"Baseline     final RMSE: mean={baseline_final.mean():.4f}  std={baseline_final.std():.4f}")
print(f"Best Variant final RMSE: mean={best_variant_final.mean():.4f}  std={best_variant_final.std():.4f}")

# Perform Mann-Whitney U test to determine if the difference is statistically significant
# We use alternative = 'less' because its a minimization problem and we want to test if the mutation variant is better than the baseline (lower RMSE)
u_stat, p_val = stats.mannwhitneyu(baseline_final, best_variant_final, alternative='less')

print(f"\nStatistical Test: Mann-Whitney U")
print(f"U-statistic: {u_stat:.4f}")
print(f"p-value:     {p_val:.5f}")
# Check if the p-value is less than the significance level (0.05)
if p_val < 0.05:
    print("There is STATISTICALLY SIGNIFICANT evidence that Mutation variant performs better than the Baseline (Mutation < Baseline).")
else:
    print("No statistical evidence of difference between Mutation and Baseline.")

# Calculate the percentage of runs that met the convergence threshold (RMSE < 30.0)
limiar = 30.0
sr_baseline     = (baseline_final < limiar).mean()
sr_best_variant = (best_variant_final < limiar).mean()

print(f"\nSuccess Rate (RMSE < {limiar})")
print(f"Baseline:     {sr_baseline:.2%}")
print(f"Best Variant: {sr_best_variant:.2%}")

Baseline     final RMSE: mean=30.4668  std=1.3379
Best Variant final RMSE: mean=30.4000  std=1.6746

Statistical Test: Mann-Whitney U
U-statistic: 214.0000
p-value:     0.65257
No statistical evidence of difference between Mutation and Baseline.

Success Rate (RMSE < 30.0)
Baseline:     40.00%
Best Variant: 55.00%


## Genetic Algorithms Vs Simulaetd Annealing

In [ ]:
# Path for the comparison between Genetic Algorithm (Baseline) and Simulated Annealing
path_ga = r"..\Project\data\results\Baseline\raw_fitness_data_50.csv"
path_sa = r"C:..\Project\data\results\Simulated_Annealing\raw_fitness_data_50.csv"

# Load the results for GA e SA
ga_data = pd.read_csv(path_ga, index_col=0)
sa_data = pd.read_csv(path_sa, index_col=0)

# Extract the final RMSE values (in the last column) for both GA and SA
ga_final = ga_data.iloc[:, -1].astype(float)
sa_final = sa_data.iloc[:, -1].astype(float)

print(f"Genetic Algorithm (Baseline) final RMSE: mean={ga_final.mean():.4f}  std={ga_final.std():.4f}")
print(f"Simulated Annealing final RMSE: mean={sa_final.mean():.4f}  std={sa_final.std():.4f}")

# Perform Mann-Whitney U test to determine if the difference is statistically significant
# We use alternative = 'less' because its a minimization problem and we want to test if the baseline (GA) is better than SA (lower RMSE)
u_stat, p_val = stats.mannwhitneyu(ga_final, sa_final, alternative='less')

print(f"\nStatistical Test: Mann-Whitney U")
print(f"U-statistic: {u_stat:.4f}")
print(f"p-value:     {p_val:.5f}")
# Check if the p-value is less than the significance level (0.05)
if p_val < 0.05:
    print("There is STATISTICALLY SIGNIFICANT evidence that GA performs better than SA (GA RMSE < SA RMSE).")
else:
    print("No statistical evidence that GA outperforms SA.")

# Calculate the percentage of runs that met the convergence threshold (RMSE < 30.0)
limiar = 30.0
sr_ga = (ga_final < limiar).mean()
sr_sa = (sa_final < limiar).mean()

print(f"\nSuccess Rate (RMSE < {limiar})")
print(f"GA: {sr_ga:.2%}")
print(f"SA: {sr_sa:.2%}")

Genetic Algorithm (Baseline) final RMSE: mean=30.4668  std=1.3379
Simulated Annealing final RMSE: mean=35.1410  std=4.0949

Statistical Test: Mann-Whitney U
U-statistic: 54.0000
p-value:     0.00004
There is STATISTICALLY SIGNIFICANT evidence that GA performs better than SA (GA RMSE < SA RMSE).

Success Rate (RMSE < 30.0)
GA: 40.00%
SA: 5.00%
